In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval transformers accelerate torch sentencepiece bitsandbytes bert_score

In [1]:
#!/usr/bin/env python3
"""
Benchmarking script for md-nishat-008/TigerLLM-9B-it (8-bit quantized)
on the Bangla CHQ-SUMM (Medical Question Summarization) dataset.
Optimized for Kaggle TPU / 2xT4 GPU with batched generation.
"""

import os
import re
import time
import traceback
import warnings
import logging
from collections import Counter

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", message=".*bitsandbytes.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from bert_score import score as bert_score


# --------------------------------------------------------------------------- #
# Device detection  (TPU → compatible GPU → CPU)
# --------------------------------------------------------------------------- #
_MIN_CUDA_CAPABILITY = (7, 0)   # bitsandbytes hard requirement


def _detect_device():
    # 1. Kaggle TPU via torch_xla
    try:
        import torch_xla.core.xla_model as xm          # noqa: F401
        print("[DEVICE] ✓ TPU detected via torch_xla — using XLA device.")
        print("         Note: 8-bit quantisation is disabled on TPU.")
        return "xla"
    except ImportError:
        pass

    # 2. CUDA GPU — only if compute capability meets bitsandbytes minimum
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        usable = []
        for i in range(count):
            name = torch.cuda.get_device_name(i)
            cap  = torch.cuda.get_device_capability(i)
            if cap >= _MIN_CUDA_CAPABILITY:
                usable.append((i, name, cap))
            else:
                print(f"[DEVICE] ⚠  GPU {i} ({name}) has compute capability "
                      f"{cap[0]}.{cap[1]} — below the sm_70 minimum for "
                      f"8-bit quantisation. Skipping.")

        if usable:
            names = [f"{n} (sm_{c[0]}{c[1]})" for _, n, c in usable]
            print(f"[DEVICE] ✓ Usable GPU(s): {', '.join(names)}")
            return "cuda"

        print("[DEVICE] ✗  No compatible GPU found (all below sm_70).")
        print("         → To use this GPU install a matching PyTorch build:")
        print("           pip install torch --index-url "
              "https://download.pytorch.org/whl/cu117")
        print("         → Falling back to CPU (slow — keep NUM_SAMPLES small).")
        return "cpu"

    # 3. CPU fallback
    print("[DEVICE] ⚠  No GPU/TPU found — falling back to CPU.")
    print("          Inference will be slow. Use a small NUM_SAMPLES first.")
    return "cpu"


DEVICE = _detect_device()


# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
MODEL_NAME = "md-nishat-008/TigerLLM-9B-it"


def find_dataset_file(filename="CHQ_SUM_dataset_2000.csv"):
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    if os.path.exists(filename):
        return filename
    raise FileNotFoundError(
        f"Could not locate {filename} under /kaggle/input or cwd."
    )


DATASET_PATH           = find_dataset_file("CHQ_SUM_dataset_2000.csv")
OUTPUT_DIR             = "/kaggle/working/tigerllm_chqsumm_outputs"
CHECKPOINT_CSV         = os.path.join(OUTPUT_DIR, "checkpoint.csv")
OUTPUT_PREDICTIONS_CSV = os.path.join(OUTPUT_DIR, "predictions.csv")
OUTPUT_RESULTS_CSV     = os.path.join(OUTPUT_DIR, "benchmark_results.csv")

BATCH_SIZE                 = 20 if DEVICE == "cuda" else 1
MAX_NEW_TOKENS             = 256
MAX_INPUT_LENGTH           = 2048
CHECKPOINT_EVERY_N_BATCHES = 5

NUM_SAMPLES   = None     # None → full 2000; set an int for a quick smoke-test
RANDOM_SAMPLE = False
SAMPLE_SEED   = 42

# Paper baseline (BanglaT5, Table 2 of the CHQSUMM paper)
PAPER_SCORES = {
    "ROUGE-1":   50.05,
    "ROUGE-2":   29.11,
    "ROUGE-L":   48.35,
    "BERTScore": 89.91,
}


# --------------------------------------------------------------------------- #
# Reasoning helpers
# --------------------------------------------------------------------------- #
_REASONING_TAGS = [
    "<think>", "</think>",
    "<reasoning>", "</reasoning>",
    "<thought>", "</thought>",
]

_reasoning_seen = {"flag": False}


def _strip_and_warn_reasoning(text: str, sample_idx: int) -> str:
    """Strip reasoning blocks; print a one-time banner + per-sample warning."""
    if not any(tag in text for tag in _REASONING_TAGS):
        return text

    if not _reasoning_seen["flag"]:
        _reasoning_seen["flag"] = True
        print("\n" + "!" * 60)
        print("  WARNING: reasoning tokens detected in model output.")
        print("  They are being stripped before metric computation.")
        print("!" * 60 + "\n")

    print(f"  [REASONING WARNING] Sample {sample_idx}: reasoning block stripped.")

    cleaned = re.sub(r"<think>.*?</think>",         "", text, flags=re.DOTALL)
    cleaned = re.sub(r"<reasoning>.*?</reasoning>", "", cleaned, flags=re.DOTALL)
    cleaned = re.sub(r"<thought>.*?</thought>",     "", cleaned, flags=re.DOTALL)
    return cleaned.strip()


def print_reasoning_status(total: int) -> None:
    print("\n" + "=" * 54)
    print("  REASONING STATUS")
    print("=" * 54)
    if not _reasoning_seen["flag"]:
        print(f"  ✓  OFF — no reasoning tokens found in {total} outputs.")
    else:
        print("  ⚠  ON  — tokens were detected and stripped.")
        print("     Summaries passed to metrics are clean.")
    print("=" * 54 + "\n")


# --------------------------------------------------------------------------- #
# Prompting
# --------------------------------------------------------------------------- #
SYSTEM_PROMPT = (
    "You are a Bengali medical question summarization model.\n"
    "Given a Bengali health-related question asked by a patient, "
    "generate a concise summary.\n"
    "Rules:\n"
    "- Retain all medically relevant information required to answer "
    "the question accurately.\n"
    "- Be as concise as possible without discarding essential information.\n"
    "- Preserve symptom details, duration, medications mentioned, and "
    "the core question.\n"
    "- Output ONLY the Bengali summary, nothing else.\n"
    "- Do not include explanations, labels, or any extra text."
)


def build_user_message(question: str) -> str:
    return f"Summarize this Bengali health question:\n{question}"


def _build_chat_prompt(tokenizer, user_message: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        messages = [
            {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_message},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


# --------------------------------------------------------------------------- #
# Model loading
# --------------------------------------------------------------------------- #
def load_model_and_tokenizer(model_name: str):
    print(f"[INFO] Loading tokenizer for {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    use_quant = DEVICE == "cuda"
    print(f"[INFO] Loading model — device={DEVICE}, 8-bit quant={use_quant} ...")

    if DEVICE == "cpu":
        print("[INFO] CPU mode: loading in float32 with low_cpu_mem_usage=True.")
        print("       This avoids peak-memory doubling during weight loading.")

    quant_config = (
        BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
        )
        if use_quant
        else None
    )

    if DEVICE == "xla":
        import torch_xla.core.xla_model as xm
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        model = model.to(xm.xla_device())
    elif DEVICE == "cpu":
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,   # stream weights in — halves peak RAM
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )

    model.eval()
    print("[INFO] Model loaded successfully.")
    return tokenizer, model


# --------------------------------------------------------------------------- #
# Batched inference
# --------------------------------------------------------------------------- #
def run_inference_batch(model, tokenizer, user_messages, max_new_tokens):
    prompts = [_build_chat_prompt(tokenizer, m) for m in user_messages]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        add_special_tokens=False,
    )

    if DEVICE == "xla":
        import torch_xla.core.xla_model as xm
        target = xm.xla_device()
    elif DEVICE == "cpu":
        target = torch.device("cpu")
    else:
        target = model.device

    inputs = {k: v.to(target) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    if DEVICE == "xla":
        import torch_xla.core.xla_model as xm
        xm.mark_step()

    input_len = inputs["input_ids"].shape[1]
    decoded = []
    for i in range(len(prompts)):
        raw = tokenizer.decode(
            output_ids[i][input_len:], skip_special_tokens=True
        ).strip()
        decoded.append(raw)
    return decoded


# --------------------------------------------------------------------------- #
# Metrics  (whitespace tokenised — Bengali-safe, no NLTK dependency)
# --------------------------------------------------------------------------- #
def _ngrams(tokens, n):
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def _rouge_n(pred_tokens, ref_tokens, n):
    pred_ng = _ngrams(pred_tokens, n)
    ref_ng  = _ngrams(ref_tokens,  n)
    overlap = sum((pred_ng & ref_ng).values())
    p = overlap / max(sum(pred_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()),  1)
    return (2 * p * r) / max(p + r, 1e-9)


def _lcs_len(a, b):
    m, n = len(a), len(b)
    prev = [0] * (n + 1)
    curr = [0] * (n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            curr[j] = (
                prev[j - 1] + 1
                if a[i - 1] == b[j - 1]
                else max(prev[j], curr[j - 1])
            )
        prev, curr = curr, [0] * (n + 1)
    return prev[n]


def _rouge_l(pred_tokens, ref_tokens):
    lcs = _lcs_len(pred_tokens, ref_tokens)
    p   = lcs / max(len(pred_tokens), 1)
    r   = lcs / max(len(ref_tokens),  1)
    return (2 * p * r) / max(p + r, 1e-9)


# --------------------------------------------------------------------------- #
# Dataset & checkpointing
# --------------------------------------------------------------------------- #
def load_dataset(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at {path}")
    df = pd.read_csv(path)
    if not {"question", "summary"}.issubset(df.columns):
        raise ValueError("Dataset must contain 'question' and 'summary' columns.")
    return df


def load_checkpoint(path: str):
    return pd.read_csv(path) if os.path.exists(path) else None


def save_checkpoint(path: str, rows: list):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")


# --------------------------------------------------------------------------- #
# Benchmark loop
# --------------------------------------------------------------------------- #
def benchmark(model, tokenizer, df: pd.DataFrame):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    existing  = load_checkpoint(CHECKPOINT_CSV)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if rows_done:
        print(f"[CHECKPOINT] ✓ Resuming — {len(rows_done)} rows already done, "
              f"starting from row {start_idx}.")
    else:
        print("[CHECKPOINT] No checkpoint found — starting from scratch.")

    if start_idx >= len(df):
        print("[INFO] All rows already complete — loaded from checkpoint.")
        return rows_done

    records     = rows_done.copy()
    num_batches = (len(df) - start_idx + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num, start in enumerate(range(start_idx, len(df), BATCH_SIZE)):
        batch_df        = df.iloc[start : start + BATCH_SIZE]
        batch_questions = batch_df["question"].tolist()
        batch_summaries = batch_df["summary"].tolist()
        batch_ids       = (
            batch_df["id"].tolist()
            if "id" in batch_df.columns
            else list(range(start, start + len(batch_df)))
        )

        prompts = [build_user_message(q) for q in batch_questions]

        t0 = time.time()
        try:
            raw_outputs = run_inference_batch(model, tokenizer, prompts, MAX_NEW_TOKENS)
            batch_error = ""
        except Exception as e:
            traceback.print_exc()
            batch_error = f"{type(e).__name__}: {e}"
            raw_outputs = [""] * len(prompts)

        batch_latency      = time.time() - t0
        latency_per_sample = batch_latency / max(len(prompts), 1)

        for i in range(len(batch_df)):
            raw_text   = raw_outputs[i] if not batch_error else ""
            global_idx = start + i

            preview = raw_text[:120] + ("…" if len(raw_text) > 120 else "")
            print(f"  [OUTPUT {global_idx}] {preview}")

            cleaned = _strip_and_warn_reasoning(raw_text, global_idx)

            records.append({
                "id":         batch_ids[i],
                "question":   batch_questions[i],
                "reference":  batch_summaries[i],
                "prediction": cleaned,
                "latency":    latency_per_sample,
                "error":      batch_error,
            })

        processed = start + len(batch_df)
        print(
            f"[INFO] Processed {processed}/{len(df)} "
            f"(batch {batch_num + 1}/{num_batches}, {batch_latency:.2f}s, "
            f"{latency_per_sample:.2f}s/sample)"
        )

        if (batch_num + 1) % CHECKPOINT_EVERY_N_BATCHES == 0:
            save_checkpoint(CHECKPOINT_CSV, records)
            print(f"[INFO] Checkpoint saved at batch {batch_num + 1} "
                  f"({len(records)} rows).")

    save_checkpoint(CHECKPOINT_CSV, records)
    save_checkpoint(OUTPUT_PREDICTIONS_CSV, records)
    print(f"[INFO] Saved predictions → {OUTPUT_PREDICTIONS_CSV}")
    return records


# --------------------------------------------------------------------------- #
# Metrics computation
# --------------------------------------------------------------------------- #
def compute_metrics(rows: list):
    predictions, references, latencies = [], [], []

    for r in rows:
        pred = str(r.get("prediction", "")).strip()
        ref  = str(r.get("reference",  "")).strip()
        err  = str(r.get("error", "")).strip().lower()
        if pred and ref and err in ("", "nan", "none"):
            predictions.append(pred)
            references.append(ref)
        try:
            latencies.append(float(r["latency"]))
        except (KeyError, TypeError, ValueError):
            pass

    if not predictions:
        print("[ERROR] No valid predictions found.")
        return {}

    print(f"[INFO] Evaluating {len(predictions)} valid predictions ...")

    r1_scores, r2_scores, rl_scores = [], [], []
    for pred, ref in zip(predictions, references):
        pt = pred.split()
        rt = ref.split()
        r1_scores.append(_rouge_n(pt, rt, 1))
        r2_scores.append(_rouge_n(pt, rt, 2))
        rl_scores.append(_rouge_l(pt, rt))

    rouge1 = sum(r1_scores) / len(r1_scores) * 100
    rouge2 = sum(r2_scores) / len(r2_scores) * 100
    rougel = sum(rl_scores) / len(rl_scores) * 100

    preds_trunc = [" ".join(p.split()[:200]) for p in predictions]
    refs_trunc  = [" ".join(r.split()[:200]) for r in references]

    print("[INFO] Computing BERTScore (xlm-roberta-base, lang=bn) ...")
    _, _, F1 = bert_score(
        preds_trunc,
        refs_trunc,
        model_type="xlm-roberta-base",
        lang="bn",
        verbose=False,
    )
    bertscore = F1.mean().item() * 100

    return {
        "ROUGE-1":    rouge1,
        "ROUGE-2":    rouge2,
        "ROUGE-L":    rougel,
        "BERTScore":  bertscore,
        "total":      len(predictions),
        "avg_latency": sum(latencies) / len(latencies) if latencies else None,
    }


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def main():
    print(f"\n[INFO] Dataset found at: {DATASET_PATH}")

    df = load_dataset(DATASET_PATH)
    print(f"[INFO] Loaded dataset with {len(df)} rows.")

    if NUM_SAMPLES is not None and NUM_SAMPLES < len(df):
        if RANDOM_SAMPLE:
            df = df.sample(n=NUM_SAMPLES, random_state=SAMPLE_SEED).reset_index(drop=True)
            print(f"[INFO] Randomly sampled {NUM_SAMPLES} rows (seed={SAMPLE_SEED}).")
        else:
            df = df.iloc[:NUM_SAMPLES].reset_index(drop=True)
            print(f"[INFO] Using first {NUM_SAMPLES} rows.")

    tokenizer, model = load_model_and_tokenizer(MODEL_NAME)

    benchmark(model, tokenizer, df)

    all_rows = load_checkpoint(CHECKPOINT_CSV).to_dict("records")
    print(f"\n[INFO] Loaded {len(all_rows)} total rows from checkpoint for metrics.")

    print_reasoning_status(len(all_rows))

    metrics = compute_metrics(all_rows)
    if not metrics:
        return

    results_df = pd.DataFrame([{
        "model":            MODEL_NAME,
        "ROUGE-1":          round(metrics["ROUGE-1"],   2),
        "ROUGE-2":          round(metrics["ROUGE-2"],   2),
        "ROUGE-L":          round(metrics["ROUGE-L"],   2),
        "BERTScore":        round(metrics["BERTScore"], 2),
        "total_sentences":  metrics["total"],
        "avg_latency_s":    round(metrics["avg_latency"], 3) if metrics["avg_latency"] else None,
    }])
    results_df.to_csv(OUTPUT_RESULTS_CSV, index=False, encoding="utf-8-sig")
    print(f"[INFO] Saved benchmark results → {OUTPUT_RESULTS_CSV}")

    print("\n── Evaluation Results (paper Table 2 format) ──")
    print(f"  {'Metric':<12} {'TigerLLM':>10}  {'BanglaT5 (paper)':>18}  {'Diff':>8}")
    print(f"  {'-'*54}")
    for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore"]:
        your  = metrics[metric]
        paper = PAPER_SCORES[metric]
        diff  = your - paper
        sign  = "+" if diff >= 0 else ""
        print(f"  {metric:<12} {your:>10.2f}  {paper:>18.2f}  {sign}{diff:>7.2f}")

    print(f"\n  Total evaluated : {metrics['total']} summaries")
    if metrics["avg_latency"]:
        print(f"  Avg latency     : {metrics['avg_latency']:.3f}s/sample")


if __name__ == "__main__":
    main()

[DEVICE] ✓ Usable GPU(s): Tesla T4 (sm_75), Tesla T4 (sm_75)

[INFO] Dataset found at: /kaggle/input/datasets/tasnifemranekanto/chq-sum/CHQ_SUM_dataset_2000.csv
[INFO] Loaded dataset with 2000 rows.
[INFO] Loading tokenizer for md-nishat-008/TigerLLM-9B-it ...
[INFO] Loading model — device=cuda, 8-bit quant=True ...


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

[INFO] Model loaded successfully.
[CHECKPOINT] ✓ Resuming — 480 rows already done, starting from row 480.
  [OUTPUT 480] মুখে, গলায়, জিহ্বায়, ঠোঁটের নিচে বা উপরে প্রায়শই ২-১টা করে বেদনাদায়ক ঘা হয়, ভিটামিন সি জাতীয় জিনিস খাচ্ছেন, কার্যকরী ওরা…
  [OUTPUT 481] সর্দি হওয়ার পর ৫ দিন ধরে গন্ধ পাচ্ছি না, ঘ্রাণ শক্তি হারিয়ে ফেলেছি কিনা।
  [OUTPUT 482] তিন মাস বয়সী বাচ্চার ৩-৪ দিন পর পর সাপোজিটরি দিয়ে পায়খানা করানো হচ্ছে, এটা দীর্ঘমেয়াদে কোনো সমস্যা কি না।
  [OUTPUT 483] জন্ডিস, হালকা শ্বাসকষ্ট, অ্যালার্জি, বুকের ডানপাশে চাপ – এই সমস্যাগুলো কয়েদিন ধরে হচ্ছে। কয়েকদিন আগে বেশি খেজুর ও মধু খে…
  [OUTPUT 484] দুইদিন ধরে বাম গালে কানের নিচে ফোলা, ডান গালে টনসিল ফোলা, গিলতে কষ্ট, মুখে লালা, জ্বর আসা-যাওয়া, গলা ফোলা এবং কাশির সাথে…
  [OUTPUT 485] দশ মাস বয়সী মেয়ে, কৃমির ওষুধ খাওয়ানো হয়নি, খাবারে অনীহা, ঔষধ খাওয়ানো উচিত কিনা।
  [OUTPUT 486] এক বছর আগে বা পায়ে চোট, ব্যাডমিন্টন খেললে বা পায়ের কোমর থেকে হাঁটু পর্যন্ত কামড়ানোর মতো ব্যথা, ক্রিকেট খেললে সমস্যা নেই,…
  [OUTPUT 487] ডান অন্ডকোষের উপরে ছো

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Saved benchmark results → /kaggle/working/tigerllm_chqsumm_outputs/benchmark_results.csv

── Evaluation Results (paper Table 2 format) ──
  Metric         TigerLLM    BanglaT5 (paper)      Diff
  ------------------------------------------------------
  ROUGE-1           28.88               50.05   -21.17
  ROUGE-2           10.27               29.11   -18.84
  ROUGE-L           26.11               48.35   -22.24
  BERTScore         89.61               89.91    -0.30

  Total evaluated : 2000 summaries
  Avg latency     : 13.933s/sample
